# Análise de sinais experimentais — janelamento, FFT e transformada de Hilbert

Este notebook usa o módulo [`signal_toolkit.py`](signal_toolkit.py) para analisar **os seus dados**.
Basta editar a célula de **configuração** abaixo com os seus arquivos e rodar tudo.

O fluxo é:
1. Sinais no domínio do tempo
2. Janelas (Hanning, Hamming, ...) no tempo e na frequência
3. Sinais janelados
4. Espectros de amplitude (FFT) sem e com janelamento
5. Amplitude, fase e frequência instantâneas (transformada de Hilbert)

In [ ]:
# Se estiver no Google Colab, descomente para baixar o módulo:
# !wget -q https://raw.githubusercontent.com/LordDolan/Signal_processing/main/signal_toolkit.py

import matplotlib.pyplot as plt
from signal_toolkit import (Sinal, plotar_sinais_tempo, plotar_sinais_janelados,
                            plotar_sinais_espectro, plotar_janelas,
                            plotar_sinais_hilbert)

## Configuração — edite aqui com os seus dados

Cada entrada da lista descreve um arquivo de aquisição:

| campo | significado |
|---|---|
| `caminho` | caminho do arquivo CSV/TXT |
| `col_tempo` / `col_sinal` | nome (ou índice) das colunas de tempo e de sinal |
| `skiprows` | linhas de cabeçalho antes da linha com os nomes das colunas |
| `t_min` / `t_max` | recorte opcional do trecho analisado (em segundos) |
| `nome` / `unidade` / `cor` | rótulos usados nos gráficos |

A taxa de amostragem é **estimada automaticamente** a partir da coluna de tempo —
não é preciso informá-la.

In [ ]:
ARQUIVOS = [
    dict(caminho="Lab_01.txt",            col_tempo="Time (s)", col_sinal="AI 1/AI 1 (N)",
         skiprows=11, t_max=0.018, nome="Harmônico",  unidade="Força (N)",        cor="blue"),
    dict(caminho="Lab_01_quadrada.txt",   col_tempo="Time (s)", col_sinal="AI 1/AI 1 (N)",
         skiprows=11, t_max=0.018, nome="Onda quadrada", unidade="Força (N)",     cor="orange"),
    dict(caminho="Lab_01_transiente.txt", col_tempo="Time (s)", col_sinal="AI 3/AI 3 (m/s2)",
         skiprows=11, t_min=2.5, t_max=4.5, nome="Transiente", unidade="Aceleração (m/s²)", cor="green"),
    dict(caminho="Lab_01_aleatorio.txt",  col_tempo="Time (s)", col_sinal="AI 1/AI 1 (N)",
         skiprows=11, t_max=1.0, nome="Aleatório", unidade="Força (N)",           cor="red"),
]

JANELAS = ["hann", "hamming"]   # janelas a comparar (aceita qualquer janela do scipy)
F_MAX = None                    # limite do eixo de frequência nos espectros (None = até Nyquist)

In [ ]:
import os
import numpy as np

if all(os.path.exists(a["caminho"]) for a in ARQUIVOS):
    sinais = [Sinal.de_arquivo(**a) for a in ARQUIVOS]
else:
    # Demonstração: se os arquivos não existirem, gera sinais sintéticos
    # equivalentes aos quatro tipos clássicos, só para o notebook rodar.
    print("Arquivos não encontrados — usando sinais sintéticos de demonstração.")
    fs, T = 1000, 2.0
    t = np.arange(0, T, 1 / fs)
    rng = np.random.default_rng(0)
    from scipy.signal import square
    sinais = [
        Sinal(t, 2.7 * np.sin(2 * np.pi * 50 * t), "Harmônico (demo)", "Força (N)", "blue"),
        Sinal(t, square(2 * np.pi * 50 * t), "Onda quadrada (demo)", "Força (N)", "orange"),
        Sinal(t, np.exp(-8 * t) * np.sin(2 * np.pi * 30 * t) * 40, "Transiente (demo)",
              "Aceleração (m/s²)", "green"),
        Sinal(t, rng.normal(size=len(t)), "Aleatório (demo)", "Força (N)", "red"),
    ]

for s in sinais:
    print(f"{s.nome}: {len(s.dados)} amostras, fs ≈ {s.fs:.1f} Hz, "
          f"t ∈ [{s.tempo.min():.3f}, {s.tempo.max():.3f}] s")

## 1. Sinais no domínio do tempo

In [ ]:
plotar_sinais_tempo(sinais)
plt.show()

## 2. Janelas no tempo e na frequência

A resposta em frequência (em dB, eixo em *bins* da FFT) mostra o compromisso de cada janela:
largura do lóbulo principal (resolução) × altura dos lóbulos laterais (vazamento espectral).

In [ ]:
plotar_janelas(JANELAS)
plt.show()

## 3. Sinais janelados no tempo

In [ ]:
for jan in JANELAS:
    plotar_sinais_janelados(sinais, jan)
plt.show()

## 4. Espectros de amplitude (FFT)

As amplitudes são normalizadas (escala 2/N) e **corrigidas pelo ganho coerente da janela**,
então uma senoide de amplitude A aparece com altura ≈ A tanto sem janela quanto com Hanning/Hamming —
os espectros são diretamente comparáveis.

In [ ]:
plotar_sinais_espectro(sinais, janela_nome=None, f_max=F_MAX)
for jan in JANELAS:
    plotar_sinais_espectro(sinais, janela_nome=jan, f_max=F_MAX)
plt.show()

## 5. Transformada de Hilbert

O sinal analítico $x_a(t) = x(t) + j\,\mathcal{H}\{x(t)\}$ fornece:

- **amplitude instantânea** $|x_a(t)|$ — o envelope do sinal;
- **fase instantânea** $\angle x_a(t)$ (desdobrada com `unwrap`);
- **frequência instantânea** $\frac{1}{2\pi}\frac{d\phi}{dt}$ — calculada com `np.gradient`, que preserva o número de pontos.

Interpretação: só faz sentido físico para sinais de banda estreita (harmônico, transiente).
Para banda larga (quadrada, aleatório) a frequência instantânea oscila fortemente.

In [ ]:
plotar_sinais_hilbert(sinais)
plt.show()

## Análise individual

Qualquer sinal pode ser analisado isoladamente com os métodos da classe `Sinal`:

In [ ]:
s = sinais[0]
s.plotar_tempo()
s.plotar_espectro("hann", f_max=F_MAX)
s.plotar_hilbert()
plt.show()

freqs, amp = s.espectro("hann")
print(f"Pico do espectro: {amp.max():.3f} {s.unidade} em {freqs[amp.argmax()]:.1f} Hz")